# SAFOD DAS + repeaters: decision checkpoint

**Decision: proceed to the nonblind development scan.**

This clean-room checkpoint has found a defensible project, but not yet a DAS catalog extension. The most interesting result is a genuine published-catalog disagreement next to a unique near-source DAS experiment: exact shared event IDs split the old target across Michel M00413 and M00414, while Michel M00414 merges events that Waldhauser--Schaff separates. The conventional array does not resolve that partition cleanly.

A second, independent opportunity is the 2026 event recorded on the deep fiber. Event 75336682 passes the frozen conventional-network verifier and is strongly visible on deep DAS; same-fiber control 75343317 fails. The family name is provisional until the catalog partition is reconciled.

This notebook reads compact products only. It does not open raw DAS, download waveforms, or alter the frozen model.

In [ ]:
# Advisor controls: edit values, then Run All.
# None uses the frozen version-1 threshold. A number is exploratory only.
from pathlib import Path
import sys

search_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT = next(
    (
        path.resolve()
        for path in search_roots
        if (path / "config" / "incremental_value.json").exists()
    ),
    None,
)
if PROJECT is None:
    raise FileNotFoundError(
        "Could not locate the standalone safod-das-repeaters repository"
    )
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

EXPLORATORY_CORRELATION_THRESHOLD = None
EXPLORATORY_LAG_RMS_THRESHOLD_S = None
REBUILD_COMPACT_CHECKPOINT = False

print("Project:", PROJECT)
print("Raw/network access: disabled in this notebook")
print("Threshold edits: exploratory; frozen products remain unchanged")

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

from src.checkpoint import write_checkpoint

if REBUILD_COMPACT_CHECKPOINT:
    checkpoint = write_checkpoint(PROJECT)
else:
    with (PROJECT / "outputs" / "checkpoint" / "advisor_checkpoint.json").open(
        "r", encoding="utf-8"
    ) as handle:
        checkpoint = json.load(handle)

display(Markdown("## Decision: {}".format(checkpoint["project_decision"])))
display(Markdown(checkpoint["decision_basis"]))

branches = pd.DataFrame(checkpoint["branches"])
milestones = pd.DataFrame(checkpoint["checkpoints"])
display(branches[["branch", "status", "evidence", "next_gate"]])
display(Markdown("### Registered checkpoints"))
display(milestones)

## 1. The label conflict is a result, not a nuisance

Labels are transferred only by exact NCSN event ID. Waldhauser--Schaff R1.2900.11955.0 maps to both Michel M00413 and M00414. Michel M00414 also contains exact IDs assigned by Waldhauser--Schaff to R1.3027.11698.0, the former hard-negative sequence.

Therefore neither “split” nor “merge” is treated as truth. This creates a focused DAS question: does the dense near-source wavefield support one partition better than the conventional array?

In [ ]:
MICHEL = PROJECT / "outputs" / "michel_validation"
with (MICHEL / "status.json").open("r", encoding="utf-8") as handle:
    michel_status = json.load(handle)
conflicts = pd.read_csv(MICHEL / "catalog_partition_conflicts.csv")
overlap = pd.read_csv(MICHEL / "sequence_overlap_matrix.csv")
crosswalk = pd.read_csv(
    MICHEL / "exact_id_crosswalk.csv", dtype={"event_id": str}
)

display(
    pd.DataFrame(
        [
            {
                "Michel events": michel_status["catalog_event_count"],
                "Michel sequences": michel_status["catalog_sequence_count"],
                "exact-ID matches": michel_status["exact_id_match_count"],
                "partition conflicts": michel_status[
                    "partition_conflict_count"
                ],
                "binary label gate": michel_status[
                    "classification_label_gate"
                ],
            }
        ]
    )
)
display(conflicts)
display(
    crosswalk.loc[
        crosswalk["michel_exact_id_match"].astype(str).str.lower().eq("true"),
        [
            "event_id",
            "waldhauser_schaff_sequence_id",
            "waldhauser_schaff_validation_role",
            "michel_sequence_id",
            "mapping_method",
        ],
    ]
)

target_id = "R1.2900.11955.0"
target_overlap = overlap.loc[
    overlap["waldhauser_schaff_sequence_id"] == target_id
].copy()
fig, ax = plt.subplots(figsize=(6.5, 3.6), constrained_layout=True)
ax.bar(
    target_overlap["michel_sequence_id"],
    target_overlap["exact_shared_event_count"],
    color=["tab:blue", "tab:orange"],
)
ax.set_ylabel("Exact shared event IDs")
ax.set_title("Published target partition across Michel sequences")
ax.grid(axis="y", alpha=0.2)
plt.show()

## 2. What the conventional array can and cannot resolve

The comparison below uses the mapped historical M00413/M00414 events and the same full conventional array used by the frozen verifier. “Within” and “between” refer to the Michel partition. Pair rows are useful diagnostics but are not independent statistical samples because events recur across pairs.

Correlation is worse than random-direction ranking for this partition (AUC 0.417). Station-centered differential lag is only modestly informative (AUC 0.630), and an event bootstrap is not identifiable with only two historical M00413 events. This is a fair comparator and a strong motivation for testing DAS spatial information—not proof that DAS wins.

In [ ]:
PARTITION = MICHEL / "partition_diagnostic"
with (PARTITION / "status.json").open("r", encoding="utf-8") as handle:
    partition_status = json.load(handle)
pair_partition = pd.read_csv(
    PARTITION / "network_pair_partition_features.csv"
)
pair_partition["relation"] = np.where(
    pair_partition["partition_relation"].str.startswith("within"),
    "within Michel sequence",
    "between M00413/M00414",
)

display(pd.DataFrame([partition_status]))
display(
    pair_partition.groupby("relation").agg(
        pair_count=("reference_event_id", "size"),
        median_correlation=("median_correlation", "median"),
        median_lag_rms_ms=("differential_lag_rms_s", lambda x: 1000 * x.median()),
    )
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
order = ["within Michel sequence", "between M00413/M00414"]
colors = ["tab:blue", "tab:orange"]
for index, (label, color) in enumerate(zip(order, colors)):
    group = pair_partition.loc[pair_partition["relation"] == label]
    jitter = np.linspace(-0.12, 0.12, len(group))
    axes[0].scatter(
        np.full(len(group), index) + jitter,
        group["median_correlation"],
        color=color,
        alpha=0.8,
    )
    axes[1].scatter(
        np.full(len(group), index) + jitter,
        1000 * group["differential_lag_rms_s"],
        color=color,
        alpha=0.8,
    )
axes[0].set_ylabel("Median full-array correlation")
axes[1].set_ylabel("Station-centered lag RMS (ms)")
for ax, title in zip(
    axes,
    ["Correlation", "Differential timing"],
):
    ax.set_xticks([0, 1], ["within", "between"])
    ax.set_title(title)
    ax.grid(alpha=0.2)
plt.show()

## 3. Frozen network verifier and threshold sandbox

Version 1 was fit on 12 historical exact-ID events before prospective waveform access. It uses two gates: median target-anchor correlation must be high, and station-centered differential-lag RMS must be low. Historical balanced accuracy is training performance, not held-out accuracy.

Set either exploratory control in the first cell to a number to see what moves. The table and figure will update, but no JSON/CSV model is changed. Because prospective outcomes are already visible, any new rule would require a version 2 and a new split.

In [ ]:
BASELINE = PROJECT / "outputs" / "incremental_value" / "network_baseline"
with (BASELINE / "network_model_frozen.json").open(
    "r", encoding="utf-8"
) as handle:
    frozen_model = json.load(handle)

correlation_threshold = (
    frozen_model["correlation_threshold"]
    if EXPLORATORY_CORRELATION_THRESHOLD is None
    else float(EXPLORATORY_CORRELATION_THRESHOLD)
)
lag_threshold_s = (
    frozen_model["differential_lag_rms_threshold_s"]
    if EXPLORATORY_LAG_RMS_THRESHOLD_S is None
    else float(EXPLORATORY_LAG_RMS_THRESHOLD_S)
)
sandbox_is_frozen = (
    EXPLORATORY_CORRELATION_THRESHOLD is None
    and EXPLORATORY_LAG_RMS_THRESHOLD_S is None
)

historical = pd.read_csv(
    BASELINE / "historical_frozen_decisions.csv", dtype={"event_id": str}
)
historical["population"] = np.where(
    historical["is_published_target"].astype(str).str.lower().eq("true"),
    "historical WS target",
    "historical WS neighbor",
)

prospective_rows = pd.read_csv(
    PROJECT
    / "outputs"
    / "incremental_value"
    / "prospective_network"
    / "frozen_network_decisions.csv",
    dtype={"event_id": str},
)
prospective_rows["population"] = "2024-25 location nominee"

continuation_rows = pd.read_csv(
    MICHEL / "frozen_network" / "decisions.csv", dtype={"event_id": str}
)
continuation_rows["population"] = (
    "Michel continuation " + continuation_rows["michel_sequence_id"]
)

deep_rows = pd.read_csv(
    PROJECT
    / "outputs"
    / "incremental_value"
    / "deep_named_network"
    / "decisions.csv",
    dtype={"event_id": str},
)
deep_rows["population"] = deep_rows["named_role"].map(
    {
        "prospective_deep_das_candidate": "2026 deep candidate",
        "prospective_same_fiber_hard_control": "2026 same-fiber control",
    }
)

scores = pd.concat(
    [historical, prospective_rows, continuation_rows, deep_rows],
    ignore_index=True,
    sort=False,
)
scores["sandbox_decision"] = np.where(
    scores["score_status"].eq("PASS")
    & (scores["median_target_correlation"] >= correlation_threshold)
    & (
        scores["median_target_differential_lag_rms_s"]
        <= lag_threshold_s
    ),
    "target_family",
    "not_target_family_or_abstain",
)

print("Frozen model SHA256:", checkpoint["network_model_sha256"])
print("Using frozen thresholds:", sandbox_is_frozen)
print("Correlation threshold:", correlation_threshold)
print("Lag-RMS threshold (s):", lag_threshold_s)
if not sandbox_is_frozen:
    print("EXPLORATORY ONLY: these values have seen prospective outcomes.")

style = {
    "historical WS target": ("tab:blue", "o"),
    "historical WS neighbor": ("tab:orange", "o"),
    "2024-25 location nominee": ("0.55", "x"),
    "Michel continuation M00413": ("tab:green", "^"),
    "Michel continuation M00414": ("tab:olive", "^"),
    "2026 deep candidate": ("tab:red", "*"),
    "2026 same-fiber control": ("tab:purple", "X"),
}
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
for population_name, group in scores.groupby("population"):
    color, marker = style.get(population_name, ("0.3", "o"))
    ax.scatter(
        group["median_target_correlation"],
        1000 * group["median_target_differential_lag_rms_s"],
        label=population_name,
        color=color,
        marker=marker,
        s=120 if "2026" in population_name else 55,
        alpha=0.85,
    )
    if (
        "historical" not in population_name
        and "location nominee" not in population_name
    ):
        for _, row in group.iterrows():
            ax.annotate(
                row["event_id"],
                (
                    row["median_target_correlation"],
                    1000 * row["median_target_differential_lag_rms_s"],
                ),
                xytext=(4, 4),
                textcoords="offset points",
                fontsize=8,
            )
ax.axvline(correlation_threshold, color="k", linestyle="--", linewidth=1)
ax.axhline(1000 * lag_threshold_s, color="k", linestyle=":", linewidth=1)
ax.set_yscale("log")
ax.set_xlabel("Median target-anchor correlation")
ax.set_ylabel("Station-centered lag RMS (ms; log scale)")
ax.set_title("Conventional verifier: target region is lower right")
ax.grid(alpha=0.2)
ax.legend(frameon=False, fontsize=8, ncol=2)
plt.show()

display(
    scores.loc[
        ~scores["population"].str.startswith("historical"),
        [
            "event_id",
            "population",
            "median_target_correlation",
            "median_target_differential_lag_rms_s",
            "frozen_decision",
            "sandbox_decision",
        ],
    ].sort_values(["population", "event_id"])
)

## 4. The 2026 deep-fiber opportunity

The joint evidence is deliberately asymmetric. Network processing was frozen before prospective access and opened zero DAS files. Separately, the pre-existing deep-DAS pilot establishes that event 75336682 is detectable across the sampled fiber, while event 75343317 supplies a same-configuration hard control.

This is stronger than “I see an earthquake on DAS,” but it is still conditional: there is only one deep candidate, the family partition is disputed, and absolute deep geometry is unsurveyed.

In [ ]:
DEEP_DAS = PROJECT / "outputs" / "deep_das"
deep_metrics = pd.read_csv(
    DEEP_DAS / "event_metrics.csv", dtype={"event_id": str}
)
with (DEEP_DAS / "hard_negative_baseline.json").open(
    "r", encoding="utf-8"
) as handle:
    hard_negative = json.load(handle)

display(
    deep_rows[
        [
            "event_id",
            "named_role",
            "median_target_correlation",
            "median_target_differential_lag_rms_s",
            "frozen_decision",
            "catalog_partition_warning",
        ]
    ]
)
display(
    deep_metrics[
        [
            "event_id",
            "role",
            "status",
            "peak_robust_z",
            "median_channel_snr",
            "detected_block_fraction",
            "usable_power_snr_ranges_hz",
        ]
    ]
)
display(pd.DataFrame([hard_negative]))

negative_figure = DEEP_DAS / "hard_negative_baseline.png"
if negative_figure.exists():
    display(Image(filename=str(negative_figure)))
else:
    print("Optional cached figure is absent:", negative_figure)

## 5. Archive population and sealed test intervals

The archive population was built from the primary manifest and an official NCSS query. Held-out intervals were selected from DAS coverage alone before catalog joining. Five routine-catalog events were nominated only by location and coverage; all five fail the frozen network target verifier.

That negative result is useful. It prevents the project from calling every nearby routine event a repeater and shows why genuine extension requires continuous DAS-only candidate generation rather than catalog proximity.

In [ ]:
INCREMENTAL = PROJECT / "outputs" / "incremental_value"
with (INCREMENTAL / "population_status.json").open(
    "r", encoding="utf-8"
) as handle:
    population_status = json.load(handle)
with (INCREMENTAL / "catalog_provenance.json").open(
    "r", encoding="utf-8"
) as handle:
    catalog_provenance = json.load(handle)

heldout = pd.read_csv(INCREMENTAL / "heldout_intervals.csv")
archive_decisions = prospective_rows[
    [
        "event_id",
        "origin_time",
        "median_target_correlation",
        "median_target_differential_lag_rms_s",
        "frozen_decision",
    ]
].copy()

display(pd.DataFrame([population_status]))
display(pd.DataFrame([catalog_provenance["manifest_stats"]]))
display(archive_decisions)
display(
    heldout[
        [
            "interval_id",
            "start_utc",
            "end_utc",
            "coverage_segment_id",
            "analysis_status",
        ]
    ]
)
print(
    "DAS waveform files opened during prospective network scoring:",
    checkpoint["guardrails"]["network_scoring_das_waveforms_opened"],
)

## 6. Next experiment and advisor decision points

The 50-minute development interval is intentionally nonblind. Its purpose is to make each detector operational and freeze its false-discovery controls—not to estimate final performance. The 12 sealed hours are opened only after both candidate generators and adjudication rules are frozen.

In [ ]:
next_experiment = pd.DataFrame(
    [
        {
            "order": 1,
            "deliverable": "Network-only continuous detector",
            "development input": "2025-01-20 04:55--05:45 UTC (50 minutes)",
            "information allowed": "network waveforms and historical templates only",
            "pass to continue": "frozen candidate table plus empirical event-level FDR",
        },
        {
            "order": 2,
            "deliverable": "DAS-only continuous detector",
            "development input": "the identical 50-minute interval",
            "information allowed": "DAS waveforms only; no network trigger times",
            "pass to continue": "frozen candidate table plus channel/block null controls",
        },
        {
            "order": 3,
            "deliverable": "Blind union adjudication",
            "development input": "deduplicated candidates",
            "information allowed": "candidate identity hidden",
            "pass to continue": "frozen event definition and ambiguity/abstention rules",
        },
        {
            "order": 4,
            "deliverable": "Held-out comparison",
            "development input": "12 sealed one-hour intervals",
            "information allowed": "both frozen pipelines",
            "pass to continue": "positive interval-bootstrap lower bound at matched FDR",
        },
    ]
)
display(next_experiment)
display(pd.DataFrame(checkpoint["project_shape"]))
print("Next analysis:", checkpoint["highest_value_next_analysis"])
print("Best next observation:", checkpoint["highest_value_next_observation"])
print("Metadata request:", checkpoint["highest_value_metadata_request"])

## 7. Source physics remains downstream

The deep audit supports channel spacing and a channel-1702 hairpin convention, not a surveyed channel-to-MD/TVD/XYZ/tangent trajectory. Therefore this checkpoint makes no absolute depth, source-distance, directivity, rupture-radius, stress-drop, or slip-rate claim.

A stress-drop branch becomes defensible only after there are at least two independently adjudicated same-configuration events, response and timing are audited, a same-path EGF ensemble exists, spectral corners lie inside the empirical usable band, and geometry/model/synthetic tests pass. A creep-rate branch additionally needs a complete recurrence population and constrained event slip.

In [ ]:
source_gates = branches.loc[
    branches["branch"].isin(
        [
            "Deep channel geometry",
            "Relative source size / stress drop",
            "Repeater-derived creep rate",
        ]
    ),
    ["branch", "status", "evidence", "next_gate"],
]
display(source_gates)

if REBUILD_COMPACT_CHECKPOINT:
    print("Compact checkpoint JSON/CSV regenerated; no raw data were opened.")
else:
    print("Cached-only advisor session complete.")